# Hand keypoints → color region growing → 64 mesh + 21 MediaPipe keypoints

Giữ nguyên **21 landmark MediaPipe** cho mỗi tay, đồng thời dùng chúng làm seed để lan vùng theo màu Lab và lấy thêm **64 điểm mesh trên đường viền**. Kết quả gồm **85 điểm mỗi tay**; các điểm mesh không thay thế landmark MediaPipe. “Mesh” ở đây là đường viền phân đoạn 2D, không phải mesh 3D MANO.

Chạy từ trên xuống, sửa `INPUT_VIDEO_DIR` và `K` (mặc định **20**). Lấy K video đầu tiên theo thứ tự tên file trong thư mục, không xếp hạng theo độ chính xác hay confidence. Mỗi video có bộ kết quả riêng theo tên file. Xuất video mask trắng/đen, video có cả 64 điểm mesh và skeleton 21 landmark, cùng NPZ tọa độ. Màu xanh/cam là mesh tay trái/phải; màu tím là skeleton và 21 điểm MediaPipe. Chất lượng vùng lan màu phụ thuộc ánh sáng, nền và che khuất. Video không chứa âm thanh.


In [1]:
from pathlib import Path
# Bỏ comment nếu kernel chưa có các thư viện này.
# %pip install mediapipe==0.10.33 opencv-python==4.11.0.86 numpy==1.26.4 tqdm
import os
import cv2
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from tqdm.auto import tqdm
from IPython.display import Video, display

from config import WLASL_RAW_DATA

ROOT = os.getcwd()
if os.path.basename(ROOT) == 'notebooks':
    ROOT = os.path.dirname(ROOT)
INPUT_VIDEO_DIR = WLASL_RAW_DATA  # ĐỔI thành thư mục video của bạn
K = 20  # Số video tối đa, lấy theo thứ tự tên file
PREVIEW_COUNT = 1  # Số video kết quả hiển thị inline; không giới hạn số video xử lý
MODEL_PATH = os.path.join(ROOT, 'pretrained', 'hand_landmarker.task')
OUTPUT_DIR = os.path.join(ROOT, 'outputs', 'hand_contour')
N_NODES = 64
COLOR_TOLERANCE = (22, 12, 12)  # L, a, b: tăng để vùng lan rộng hơn
ROI_PADDING = 0.18  # theo kích thước bbox landmarks
SUPPORT_RADIUS = 0.18  # theo chiều dài lòng bàn tay; giới hạn lan khỏi skeleton
MAX_FRAMES = None  # ví dụ 150 để chạy thử; None = toàn bộ video
SHOW_NODE_IDS = False


def select_video_paths(directory, k):
    if isinstance(k, bool) or not isinstance(k, int) or k < 1:
        raise ValueError('K phải là số nguyên dương')
    directory = Path(directory)
    if not directory.is_dir():
        raise FileNotFoundError(f'Sửa INPUT_VIDEO_DIR: {directory}')
    extensions = {'.mp4', '.avi', '.mov', '.mkv', '.webm', '.m4v'}
    videos = sorted((path for path in directory.iterdir()
                     if path.is_file() and path.suffix.lower() in extensions),
                    key=lambda path: (path.name.casefold(), path.name))
    if not videos:
        raise FileNotFoundError(f'Không tìm thấy video trong {directory}')
    # Tránh trùng tên output khi hai file chỉ khác đuôi video.
    selected = videos[:k]
    stems = [path.stem.casefold() for path in selected]
    if len(stems) != len(set(stems)):
        raise ValueError('Các video được chọn có tên gốc trùng nhau; đổi tên trước khi chạy')
    print(f'Chọn {len(selected)}/{len(videos)} video (K={k}):')
    for index, video in enumerate(selected, 1):
        print(f'{index:02d}. {video.name}')
    return selected


C:\Users\ADMIN\miniconda3\envs\signdetection\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Lan màu từ 21 seed, giữ riêng landmark gốc
Dùng flood fill **fixed range** trong không gian Lab, giới hạn bởi ROI và vùng hỗ trợ skeleton. Các seed được làm tròn và giới hạn trong ROI **chỉ để phân đoạn**; 21 landmark gốc không bị thay đổi bởi bước này. Hợp nhất các vùng lan màu, chọn đường viền chứa nhiều seed nhất rồi lấy đều 64 điểm theo độ dài đường viền. Ghép theo thứ tự `[64 mesh, 21 MediaPipe]` sau khi đổi cả hai về cùng hệ tọa độ pixel ảnh gốc.


In [2]:
EDGES = [(0,1),(1,2),(2,3),(3,4),(0,5),(5,6),(6,7),(7,8),
         (5,9),(9,10),(10,11),(11,12),(9,13),(13,14),(14,15),(15,16),
         (13,17),(0,17),(17,18),(18,19),(19,20)]

def segment_hand(frame, points):
    h, w = frame.shape[:2]
    span = max(float(np.ptp(points[:,0])), float(np.ptp(points[:,1])), 10.)
    pad = max(4, int(span * ROI_PADDING))
    x0, y0 = np.maximum(np.floor(points.min(0)).astype(int)-pad, 0)
    x1, y1 = np.minimum(np.ceil(points.max(0)).astype(int)+pad+1, [w,h])
    full = np.zeros((h,w), np.uint8)
    if x1 <= x0 or y1 <= y0:
        return full
    crop = frame[y0:y1, x0:x1]
    pts = np.rint(points-[x0,y0]).astype(int)
    pts[:,0] = np.clip(pts[:,0],0,crop.shape[1]-1)
    pts[:,1] = np.clip(pts[:,1],0,crop.shape[0]-1)
    support = np.zeros(crop.shape[:2], np.uint8)
    radius = max(3, int(np.linalg.norm(points[0]-points[9])*SUPPORT_RADIUS))
    for a,b in EDGES:
        cv2.line(support, tuple(pts[a]), tuple(pts[b]),255,2*radius)
    palm = cv2.convexHull(pts[[0,1,5,9,13,17]])
    cv2.fillConvexPoly(support,palm,255)
    support = cv2.dilate(support, np.ones((3,3),np.uint8))
    lab = cv2.cvtColor(cv2.GaussianBlur(crop,(3,3),0),cv2.COLOR_BGR2LAB)
    union = np.zeros(crop.shape[:2],np.uint8)
    flags = 4 | cv2.FLOODFILL_FIXED_RANGE | cv2.FLOODFILL_MASK_ONLY | (255 << 8)
    for x,y in pts:
        # Mỗi seed dùng mask độc lập, không chặn lẫn nhau.
        flood_mask = np.zeros((crop.shape[0]+2,crop.shape[1]+2),np.uint8)
        flood_mask[1:-1,1:-1][support==0] = 1
        image = lab.copy()
        patch = lab[max(0,y-1):y+2,max(0,x-1):x+2]
        image[y,x] = np.median(patch.reshape(-1,3),axis=0).astype(np.uint8)
        cv2.floodFill(image,flood_mask,(int(x),int(y)),(0,0,0),
                      COLOR_TOLERANCE,COLOR_TOLERANCE,flags)
        union[flood_mask[1:-1,1:-1]==255] = 255
    union = cv2.morphologyEx(union,cv2.MORPH_CLOSE,np.ones((3,3),np.uint8))
    union[support==0] = 0
    contours,_ = cv2.findContours(union,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_NONE)
    if contours:
        def score(c):
            hits = sum(cv2.pointPolygonTest(c,(float(x),float(y)),False)>=0 for x,y in pts)
            return hits,cv2.contourArea(c)
        best = max(contours,key=score)
        if cv2.contourArea(best)>=10:
            clean = np.zeros_like(union)
            cv2.drawContours(clean,[best],-1,255,cv2.FILLED)
            full[y0:y1,x0:x1] = clean
    return full


def contour_nodes(mask, wrist, previous=None):
    contours,_ = cv2.findContours(mask,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_NONE)
    if not contours:
        return None
    p = max(contours,key=cv2.contourArea)[:,0,:].astype(np.float32)
    if len(p)<3:
        return None
    # Hướng cố định và node đầu gần cổ tay ở frame đầu.
    area = np.sum(p[:,0]*np.roll(p[:,1],-1)-np.roll(p[:,0],-1)*p[:,1])
    if area<0:
        p = p[::-1]
    p = np.roll(p,-int(np.argmin(np.linalg.norm(p-wrist,axis=1))),axis=0)
    closed = np.vstack([p,p[:1]])
    lengths = np.linalg.norm(np.diff(closed,axis=0),axis=1)
    keep = np.r_[True,lengths>1e-6]
    closed = closed[keep]
    distance = np.r_[0,np.cumsum(np.linalg.norm(np.diff(closed,axis=0),axis=1))]
    if distance[-1]<=0:
        return None
    samples = np.linspace(0,distance[-1],N_NODES,endpoint=False)
    nodes = np.column_stack([np.interp(samples,distance,closed[:,i]) for i in range(2)]).astype(np.float32)
    if previous is not None:
        # Căn cyclic để giảm nhảy chỉ số giữa hai frame liên tiếp.
        shifts = [np.roll(nodes,k,axis=0) for k in range(N_NODES)]
        nodes = min(shifts,key=lambda n:np.mean((n-previous)**2))
    return nodes


def combine_hand_keypoints(contour_xy, landmark_xyz, image_size):
    """Combine (..., 64, 2) mesh and (..., 21, 3) original landmarks."""
    scale = np.asarray(image_size, dtype=np.float32)
    if scale.shape != (2,) or not np.isfinite(scale).all() or np.any(scale <= 0):
        raise ValueError('image_size must contain positive width and height')
    contour_xy = np.asarray(contour_xy, dtype=np.float32)
    landmark_xyz = np.asarray(landmark_xyz, dtype=np.float32)
    if (contour_xy.shape[-2:] != (N_NODES, 2)
            or landmark_xyz.shape[-2:] != (21, 3)
            or contour_xy.shape[:-2] != landmark_xyz.shape[:-2]):
        raise ValueError('Expected matching mesh (..., 64, 2) and landmarks (..., 21, 3)')
    landmark_xy = landmark_xyz[..., :2] * scale
    combined = np.concatenate([contour_xy, landmark_xy], axis=-2)
    valid = np.isfinite(combined).all(axis=-1)
    return combined, combined / scale, valid


def draw_mediapipe_keypoints(image, points):
    """Draw all original landmarks even when color segmentation fails."""
    color = (255, 0, 255)
    visible = np.isfinite(points).all(axis=-1)
    for a, b in EDGES:
        if visible[a] and visible[b]:
            cv2.line(image, tuple(np.rint(points[a]).astype(int)),
                     tuple(np.rint(points[b]).astype(int)), color, 1)
    for j in np.flatnonzero(visible):
        x, y = np.rint(points[j]).astype(int)
        cv2.circle(image, (x, y), 3, color, -1)
        if SHOW_NODE_IDS:
            cv2.putText(image, f'MP{j}', (x + 3, y - 3),
                        cv2.FONT_HERSHEY_SIMPLEX, .3, color, 1)


## Chạy lần lượt tối đa K video
NPZ lưu các mảng theo frame và slot tay (0 = Left, 1 = Right theo MediaPipe):

- `keypoints_xy`: `(T, 2, 85, 2)`, tọa độ pixel ảnh gốc; 0–63 là mesh, 64–84 là MediaPipe.
- `keypoints_xy_normalized`: cùng shape, x chia chiều rộng và y chia chiều cao ảnh.
- `keypoint_valid`: `(T, 2, 85)`, đánh dấu từng điểm hợp lệ.
- `contour_xy`: `(T, 2, 64, 2)`, giữ tên cũ cho 64 điểm mesh.
- `landmark_xyz`: `(T, 2, 21, 3)`, tọa độ normalized MediaPipe gốc, giữ nguyên cả z.
- `mesh_valid` và `landmark_valid`: `(T, 2)`, trạng thái độc lập; `valid` là alias của `mesh_valid` để tương thích dữ liệu cũ.

Điểm thiếu được ghi NaN. Khi mask không tạo được mesh, 21 điểm MediaPipe vẫn được lưu và vẽ nếu phát hiện được. Không điền điểm bằng frame trước. Chỉ số mesh lấy mẫu theo đường viền, chưa phải correspondence giải phẫu ổn định.

Mỗi video dùng một tracker mới và lưu hai MP4 cùng một NPZ riêng. Video lỗi được ghi trong `failed_videos`; các video còn lại vẫn tiếp tục. Danh sách kết quả thành công nằm trong `video_results`.


In [3]:
def process_video(input_video):
    if not os.path.isfile(input_video):
        raise FileNotFoundError(f'Không tìm thấy video: {input_video}')
    if not os.path.isfile(MODEL_PATH):
        raise FileNotFoundError(f'Thiếu MediaPipe model: {MODEL_PATH}')
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    cap = cv2.VideoCapture(str(input_video))
    if not cap.isOpened():
        cap.release()
        raise RuntimeError(f'Không mở được video: {input_video}')
    writers = []
    all_nodes, all_landmarks, all_valid, times = [], [], [], []
    previous = [None,None]
    mesh_path = os.path.join(OUTPUT_DIR, f'{os.path.splitext(os.path.basename(input_video))[0]}_hand_mesh_bw.mp4')
    node_path = os.path.join(OUTPUT_DIR, f'{os.path.splitext(os.path.basename(input_video))[0]}_contour_{N_NODES}.mp4')
    try:
        fps = cap.get(cv2.CAP_PROP_FPS)
        if not np.isfinite(fps) or fps<=0:
            fps = 30.
        w,h = int(cap.get(3)),int(cap.get(4))
        # mp4v cần kích thước chẵn; pad cạnh phải/dưới nếu cần.
        size = (w+w%2,h+h%2)
        for path in (mesh_path,node_path):
            writer = cv2.VideoWriter(str(path),cv2.VideoWriter_fourcc(*'mp4v'),fps,size)
            writers.append(writer)
            if not writer.isOpened():
                raise RuntimeError(f'Không tạo được video: {path}')
        options = vision.HandLandmarkerOptions(
            base_options=python.BaseOptions(model_asset_path=str(MODEL_PATH)),
            running_mode=vision.RunningMode.VIDEO,num_hands=2,
            min_hand_detection_confidence=0.5,min_hand_presence_confidence=0.5,
            min_tracking_confidence=0.5)
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        limit = min(total,MAX_FRAMES) if MAX_FRAMES is not None and total>0 else (MAX_FRAMES or total or None)
        with vision.HandLandmarker.create_from_options(options) as detector, tqdm(total=limit) as progress:
            index, last_ms = 0,-1
            while MAX_FRAMES is None or index<MAX_FRAMES:
                ok,frame = cap.read()
                if not ok:
                    break
                ms = max(last_ms+1,round(index*1000/fps))
                last_ms = ms
                result = detector.detect_for_video(mp.Image(image_format=mp.ImageFormat.SRGB,
                    data=np.ascontiguousarray(cv2.cvtColor(frame,cv2.COLOR_BGR2RGB))),ms)
                nodes_frame = np.full((2,N_NODES,2),np.nan,np.float32)
                landmarks_frame = np.full((2,21,3),np.nan,np.float32)
                valid = np.zeros(2,bool)
                bw = np.zeros((h,w),np.uint8)
                overlay = frame.copy()
                occupied = set()
                # Ưu tiên dự đoán handedness có confidence cao khi trùng nhãn.
                order = sorted(range(len(result.hand_landmarks)),key=lambda i:result.handedness[i][0].score,reverse=True)
                for i in order:
                    slot = 0 if result.handedness[i][0].category_name=='Left' else 1
                    if slot in occupied:
                        continue
                    occupied.add(slot)
                    xyz = np.array([(p.x,p.y,p.z) for p in result.hand_landmarks[i]],np.float32)
                    landmarks_frame[slot] = xyz
                    points = xyz[:,:2]*[w,h]
                    draw_mediapipe_keypoints(overlay, points)
                    mask = segment_hand(frame,points)
                    nodes = contour_nodes(mask,points[0],previous[slot])
                    bw = cv2.bitwise_or(bw,mask)
                    if nodes is None:
                        continue
                    nodes_frame[slot],valid[slot] = nodes,True
                    color = (0,255,0) if slot==0 else (0,180,255)
                    for j,(x,y) in enumerate(np.rint(nodes).astype(int)):
                        cv2.circle(overlay,(x,y),2,color,-1)
                        if SHOW_NODE_IDS:
                            cv2.putText(overlay,f'C{j}',(x+2,y),cv2.FONT_HERSHEY_SIMPLEX,.3,color,1)
                previous = [nodes_frame[s].copy() if valid[s] else None for s in range(2)]
                for writer,img in zip(writers,[cv2.cvtColor(bw,cv2.COLOR_GRAY2BGR),overlay]):
                    writer.write(cv2.copyMakeBorder(img,0,size[1]-h,0,size[0]-w,cv2.BORDER_CONSTANT))
                all_nodes.append(nodes_frame)
                all_landmarks.append(landmarks_frame)
                all_valid.append(valid)
                times.append(ms)
                index += 1
                progress.update(1)
        if not all_nodes:
            raise RuntimeError('Video không có frame đọc được.')
        data_path = os.path.join(OUTPUT_DIR, f'{os.path.splitext(os.path.basename(input_video))[0]}_contour_{N_NODES}.npz')
        contour_xy = np.stack(all_nodes)
        landmark_xyz = np.stack(all_landmarks)
        keypoints_xy, keypoints_normalized, keypoint_valid = combine_hand_keypoints(
            contour_xy, landmark_xyz, [w, h])
        mesh_valid = np.stack(all_valid)
        landmark_valid = np.isfinite(landmark_xyz).all(axis=(-2, -1))
        np.savez_compressed(data_path,contour_xy=contour_xy,
            landmark_xyz=landmark_xyz,valid=mesh_valid,
            mesh_valid=mesh_valid,landmark_valid=landmark_valid,
            keypoints_xy=keypoints_xy,keypoints_xy_normalized=keypoints_normalized,
            keypoint_valid=keypoint_valid,
            keypoint_source=np.array(['mesh'] * N_NODES + ['mediapipe'] * 21),
            keypoint_index=np.r_[np.arange(N_NODES), np.arange(21)],
            timestamp_ms=np.asarray(times),fps=fps,image_size=np.array([w,h]),
            hand_slots=np.array(['Left','Right']))
    finally:
        cap.release()
        for writer in writers:
            writer.release()
    print(f'Frames: {len(times)}; valid Left/Right: {np.sum(all_valid,axis=0)}')
    print(f'Keypoints per hand: {N_NODES} mesh + 21 MediaPipe = {N_NODES + 21}')
    print(f'keypoints_xy: {keypoints_xy.shape}; landmark_xyz: {landmark_xyz.shape}')
    print(mesh_path, node_path, data_path, sep='\n')
    return mesh_path,node_path,data_path

def process_videos(input_videos):
    if not os.path.isfile(MODEL_PATH):
        raise FileNotFoundError(f'Thiếu MediaPipe model: {MODEL_PATH}')
    results, failures = [], []
    for index, video in enumerate(input_videos, 1):
        print(f'\n[{index}/{len(input_videos)}] {video}')
        try:
            mesh_video, contour_video, keypoint_file = process_video(video)
        except (OSError, RuntimeError, ValueError) as error:
            failures.append({'input_video': str(video), 'error': str(error)})
            print(f'Lỗi: {error}')
            continue
        results.append({'input_video': str(video), 'mesh_video': mesh_video,
                        'contour_video': contour_video, 'keypoint_file': keypoint_file})
    print(f'\nHoàn tất: {len(results)} thành công, {len(failures)} lỗi.')
    return results, failures


input_videos = select_video_paths(INPUT_VIDEO_DIR, K)
video_results, failed_videos = process_videos(input_videos)


Chọn 20/12329 video (K=20):
01. 00335.mp4
02. 00336.mp4
03. 00338.mp4
04. 00339.mp4
05. 00341.mp4
06. 00376.mp4
07. 00377.mp4
08. 00381.mp4
09. 00382.mp4
10. 00384.mp4
11. 00414.mp4
12. 00415.mp4
13. 00416.mp4
14. 00421.mp4
15. 00426.mp4
16. 00430.mp4
17. 00431.mp4
18. 00433.mp4
19. 00435.mp4
20. 00583.mp4

[1/20] D:\SignDetection\datasets\raw\WLASL\videos\00335.mp4


100%|██████████| 58/58 [00:03<00:00, 17.31it/s]


Frames: 58; valid Left/Right: [ 0 24]
Keypoints per hand: 64 mesh + 21 MediaPipe = 85
keypoints_xy: (58, 2, 85, 2); landmark_xyz: (58, 2, 21, 3)
C:\Users\ADMIN\OneDrive\Desktop\projects\sign-language-recognition\outputs\hand_contour\00335_hand_mesh_bw.mp4
C:\Users\ADMIN\OneDrive\Desktop\projects\sign-language-recognition\outputs\hand_contour\00335_contour_64.mp4
C:\Users\ADMIN\OneDrive\Desktop\projects\sign-language-recognition\outputs\hand_contour\00335_contour_64.npz

[2/20] D:\SignDetection\datasets\raw\WLASL\videos\00336.mp4


100%|██████████| 65/65 [00:16<00:00,  3.92it/s]


Frames: 65; valid Left/Right: [ 0 54]
Keypoints per hand: 64 mesh + 21 MediaPipe = 85
keypoints_xy: (65, 2, 85, 2); landmark_xyz: (65, 2, 21, 3)
C:\Users\ADMIN\OneDrive\Desktop\projects\sign-language-recognition\outputs\hand_contour\00336_hand_mesh_bw.mp4
C:\Users\ADMIN\OneDrive\Desktop\projects\sign-language-recognition\outputs\hand_contour\00336_contour_64.mp4
C:\Users\ADMIN\OneDrive\Desktop\projects\sign-language-recognition\outputs\hand_contour\00336_contour_64.npz

[3/20] D:\SignDetection\datasets\raw\WLASL\videos\00338.mp4


...

Frames: 79; valid Left/Right: [79 76]
Keypoints per hand: 64 mesh + 21 MediaPipe = 85
keypoints_xy: (79, 2, 85, 2); landmark_xyz: (79, 2, 21, 3)
C:\Users\ADMIN\OneDrive\Desktop\projects\sign-language-recognition\outputs\hand_contour\00433_hand_mesh_bw.mp4
C:\Users\ADMIN\OneDrive\Desktop\projects\sign-language-recognition\outputs\hand_contour\00433_contour_64.mp4
C:\Users\ADMIN\OneDrive\Desktop\projects\sign-language-recognition\outputs\hand_contour\00433_contour_64.npz

[19/20] D:\SignDetection\datasets\raw\WLASL\videos\00435.mp4


100%|██████████| 103/103 [00:07<00:00, 14.42it/s]


Frames: 103; valid Left/Right: [46 39]
Keypoints per hand: 64 mesh + 21 MediaPipe = 85
keypoints_xy: (103, 2, 85, 2); landmark_xyz: (103, 2, 21, 3)
C:\Users\ADMIN\OneDrive\Desktop\projects\sign-language-recognition\outputs\hand_contour\00435_hand_mesh_bw.mp4
C:\Users\ADMIN\OneDrive\Desktop\projects\sign-language-recognition\outputs\hand_contour\00435_contour_64.mp4
C:\Users\ADMIN\OneDrive\Desktop\projects\sign-language-recognition\outputs\hand_contour\00435_contour_64.npz

[20/20] D:\SignDetection\datasets\raw\WLASL\videos\00583.mp4


100%|██████████| 38/38 [00:05<00:00,  6.77it/s]

Frames: 38; valid Left/Right: [ 1 24]
Keypoints per hand: 64 mesh + 21 MediaPipe = 85
keypoints_xy: (38, 2, 85, 2); landmark_xyz: (38, 2, 21, 3)
C:\Users\ADMIN\OneDrive\Desktop\projects\sign-language-recognition\outputs\hand_contour\00583_hand_mesh_bw.mp4
C:\Users\ADMIN\OneDrive\Desktop\projects\sign-language-recognition\outputs\hand_contour\00583_contour_64.mp4
C:\Users\ADMIN\OneDrive\Desktop\projects\sign-language-recognition\outputs\hand_contour\00583_contour_64.npz

Hoàn tất: 20 thành công, 0 lỗi.


In [4]:
# Một số browser không hỗ trợ codec mp4v; khi đó mở MP4 bằng VLC.
# Tăng PREVIEW_COUNT để xem thêm kết quả, hoặc chọn trực tiếp video_results[i].
for result in video_results[:PREVIEW_COUNT]:
    print(result['input_video'])
    display(Video(str(result['mesh_video']), embed=True, width=640))
    display(Video(str(result['contour_video']), embed=True, width=640))
if failed_videos:
    print('Video bị lỗi:')
    for failure in failed_videos:
        print(f"{failure['input_video']}: {failure['error']}")


D:\SignDetection\datasets\raw\WLASL\videos\00335.mp4
